# Backends


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## StateBackend

In [4]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


agent = create_deep_agent(model='groq:llama-3.3-70b-versatile')
# default backend = StateBackend()

agent2 = create_deep_agent(
    model='groq:llama-3.3-70b-versatile',
    backend=StateBackend()
)

In [ ]:
"""
Invoke agent and ask it to WRITE to a file 
(StateBackend keeps the file inside the LangGraph state)
"""

result = agent2.invoke({
    'messages':[{
        'role': 'user',
        'content': (
            'create a file at /notes/todo.txt with the below content:\n'
            '1. learn fastapi\n2.learn a frontend framework\n3.make a project'
            'Let me know when done'
        )
    }]
})

print('----Agent Reply----')
print(result['messages'][-1].content)

----Agenbt Reply----
The file has been created at /notes/todo.txt with the specified content.


In [7]:
# check if backend is working 
print('----Backend Check-----')
files = result.get('files', {})

if files:
    print(f'stateBackend is working - {len(files)} files in state:')
    for path, content in files.items():
        print(f'\n {path}\n{'-' *40}\n{content}')
else:
    print('No files found in state')

----Backend Check-----
stateBackend is working - 1 files in state:

 /notes/todo.txt
----------------------------------------
{'content': '1. learn fastapi\n2. learn a frontend framework\n3. make a project', 'encoding': 'utf-8', 'created_at': '2026-07-20T10:15:03.926599+00:00', 'modified_at': '2026-07-20T10:15:03.926599+00:00'}


In [8]:
# Proving persistence within same thread

followup = agent2.invoke({
    'messages': result['messages'] + [{
        'role': 'user',
        'content': "Read /notes/todo.txt and print it's content"
    }],
    'files': result.get('files', {})    # passing virtual filesystem 
})

print('----Read back (same thread)----')
print(followup['messages'][-1].content)

----Read back (same thread)----
The content of /notes/todo.txt is:
1. learn fastapi
2. learn a frontend framework
3. make a project


## FileSystemBackend (local disk)

In [14]:
'''
Create the agent with real disk backend
root_dir = '.' -> files land realtive to current working directory
virtual_mode = True -> agent uses virtual paths like /notes/todo.txt mapped onto root_dir
'''

from deepagents.backends import FilesystemBackend

ROOT = "."
agent = create_deep_agent(
    model='groq:qwen/qwen3.6-27b',
    backend=FilesystemBackend(
        root_dir=ROOT,
        virtual_mode=True
    )
)


In [15]:
# invoking agent and asking it to WRITE a file

result = agent.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            'create a file at /notes/todo.txt with the below content:\n'
            '1.Go home\n2.Cook dinner\n3.Go for midnight walk\n'
            'Let me know onces done.'
        )
    }]
})

print('----Agent Reply----')
print(result['messages'][-1].content)

----Agent Reply----
File updated at `/notes/todo.txt` with the content:

1.Go home
2.Cook dinner
3.Go for midnight walk
